# Approach 2 — B0–A6: Add-on Ablation + DINOv2 Fine-tuning Depth

**Based on the uploaded `Approach_2_Matched_Selection(1).ipynb` and the accompanying fine-tuning depth experiment.**

This is **two controlled comparisons, not seven unrelated model designs**:

| ID | Decor (logit) | Cross-crop SupCon | Extra Gaussian Blur | Unfrozen DINOv2 Blocks | What changes? |
|:---|:---:|:---:|:---:|---:|---|
| **B0** Baseline2 | ✗ | ✗ | ✗ | 0 | Architecture/training control |
| **A1** | ✓ | ✗ | ✗ | 0 | Decorrelation addition |
| **A2** | ✓ | ✓ | ✗ | 0 | SupCon addition |
| **A3** | ✓ | ✓ | ✓ | 0 | Extra augmentation addition |
| **A4** | ✓ | ✓ | ✓ | 2 | Fine-tune last two Blocks |
| **A5** | ✓ | ✓ | ✓ | 4 | Fine-tune last four Blocks |
| **A6** | ✓ | ✓ | ✓ | 6 | Fine-tune last six Blocks |

> **Interpretation:** B0→A1→A2→A3 measures *additive* improvements; A3→A4/A5/A6 isolates fine-tuning depth **conditional on all A3 add-ons being enabled**. A1/A2 together do **not** establish a complete factorial interaction analysis. Decorrelation and SupCon here operate on centered **classification logits**, not distinct projected embeddings. These should not be described as proven feature disentanglement.

**Evaluation protocol:** Only training CSV is used for training, epoch/recipe selection and calibration tuning. Five training-derived proxy pairs are used for selection; five independent training-derived proxy pairs are held for a check. After development selection is frozen, each recipe is freshly refit on all 8,000 training images. Official evaluation is a separately enabled step; use 323 seen + 10 unseen pair identities, generate predictions, and **only then load per-image official ground-truth labels**. Raw and calibrated results are reported separately.

**Hardware note:** The FT4 and FT6 recipes fine-tune a substantial portion of a ViT-B/14 and can take considerably longer or run out of VRAM on laptop GPUs. Automatic mixed precision is uniformly enabled for all recipes on CUDA, but actual training time/memory depends on your machine. If you must change batch size, change it for **all** recipes and document that it changes the experiment.

**Important:** Since previous official test results were already inspected in the project, do not use additional official test scores to repeatedly tune/select these methods. Decide using train-derived development and report this limitation honestly.


In [ ]:
DEBUG = True                         # True: short 1-epoch smoke test; False: full experiment
USE_AMP = True                      # Uniform CUDA AMP for all B0–A6 runs (False for strict FP32)
RUN_FULL_REFIT = True               # When DEBUG=False, refit ALL seven selected recipes on 8,000 training rows

from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import sys
import gc

import pandas as pd
import timm
import torch
from IPython.display import display
from PIL import Image
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import functional as TF

here = Path.cwd()
ROOT = next((p for p in (here, *here.parents) if (p / 'common' / 'dataset.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Open this notebook inside the team GitHub repository containing common/dataset.py.')
sys.path.insert(0, str(ROOT))

from common import config as team
from common.dataset import PlantDataset, get_loaders, get_pairs, get_transforms, load_split
from common.evaluate import PERTURBATIONS, harmonic_mean, save_confusion, top_k_accuracy

BACKBONE = 'vit_base_patch14_dinov2.lvd142m'
REFERENCE_COMMIT = '0fece2daff4917431bb2a5b16b65718ab100d05c'
BATCH_SIZE = 8 if DEBUG else 16
EPOCHS = 1 if DEBUG else 20
DEVICE = team.DEVICE
AMP = bool(USE_AMP and torch.cuda.is_available())
NUM_CROPS, NUM_DISEASES = team.NUM_CROPS, team.NUM_DISEASES
team.NUM_WORKERS = 0
RUN_NAME = 'approach2_b0_a6_ablation_depth' + ('_debug' if DEBUG else '')
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT_DIR = team.RESULTS_DIR / RUN_NAME / RUN_ID
DEVELOPMENT_DIR = OUTPUT_DIR / 'development'
FINAL_DIR = OUTPUT_DIR / 'final'
OFFICIAL_DIR = OUTPUT_DIR / 'official' / 'pair_identity_order'
CHECKPOINT_DIR = team.CHECKPOINT_DIR / RUN_NAME / RUN_ID

HEAD_LR, BACKBONE_LR, WEIGHT_DECAY = 1e-4, 1e-5, 1e-4
SMOOTHING = 0.1
DECOR_WEIGHT, SUPCON_WEIGHT, TEMPERATURE = 0.05, 0.10, 0.10
BLUR_PROB, BLUR_SIGMA = 0.15, 0.7
GAMMAS = [step / 2 for step in range(25)]
RECIPES = {
    'B0_Baseline2': dict(decor=0.0, supcon=0.0, blur=False, depth=0),
    'A1_Decorrelation': dict(decor=DECOR_WEIGHT, supcon=0.0, blur=False, depth=0),
    'A2_Add_SupCon': dict(decor=DECOR_WEIGHT, supcon=SUPCON_WEIGHT, blur=False, depth=0),
    'A3_Add_Blur': dict(decor=DECOR_WEIGHT, supcon=SUPCON_WEIGHT, blur=True, depth=0),
    'A4_FT2': dict(decor=DECOR_WEIGHT, supcon=SUPCON_WEIGHT, blur=True, depth=2),
    'A5_FT4': dict(decor=DECOR_WEIGHT, supcon=SUPCON_WEIGHT, blur=True, depth=4),
    'A6_FT6': dict(decor=DECOR_WEIGHT, supcon=SUPCON_WEIGHT, blur=True, depth=6),
}
BASELINE = 'B0_Baseline2'
PROPOSALS = tuple(r for r in RECIPES if r != BASELINE)
FINE_TUNES = ('A4_FT2','A5_FT4','A6_FT6')

if not team.TRAIN_CSV.is_file() or not team.IMAGE_DIR.is_dir():
    raise FileNotFoundError(f'Missing train.csv or images under {team.DATA_DIR}')
if (team.NUM_SEEN_PAIRS, team.NUM_UNSEEN_PAIRS, team.NUM_PAIRS) != (323, 10, 333):
    raise ValueError('Use the team 323-seen / 333-total pairing conventions.')
print(f'{DEVICE} | CUDA AMP={AMP} | batch={BATCH_SIZE} | epochs={EPOCHS} | debug={DEBUG}')
print('Run:', RUN_ID)


## 2. Data: one shared project split + train-derived proxy-unseen validation

`B0` uses the common within-pair **fit/validation** split and original **seen validation Top-1** for checkpoint selection. A1–A6 train their **development copies** on the same fit rows **after excluding ten entire training-derived pairs**. The held-out pairs contain concepts that appear elsewhere in the fit subset. Five pairs are used to select epoch, recipe and γ; the other five are used once for independent checking. Those ten rows are **not** official unseen test combinations. All recipes use the original training transform and concept labels. The independent check does not select hyperparameters.

**Fairness caveat:** B0 uses the original common split for its development training/selection; A1–A6 use the same proxy-holdout fit subset to make zero-shot model selection meaningful. All A1–A6 share the same proxy-development fit and selection procedure, but their selected epochs may differ. The **full refit** stage equalizes the final training set (all 8,000 train images); B0 uses the original seen-validation selection rule by design. This is more defensible than treating seen-val accuracy as a proxy for zero-shot generalisation.


In [ ]:
def image_key(value):
    result = Path(str(value).strip().replace("\\", "/"))
    if not result.name or result.is_absolute() or ".." in result.parts or ":" in str(result):
        raise ValueError(f"Expected a relative image filename, got {value!r}.")
    return result.as_posix()


def sha256_file(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def atomic_json(path, value):
    temp = path.with_suffix(".json.tmp")
    temp.write_text(json.dumps(value, indent=2), encoding="utf-8")
    temp.replace(path)


def atomic_save(path, value):
    temp = path.with_suffix(".pt.tmp")
    torch.save(value, temp)
    temp.replace(path)


train_full = load_split(team.TRAIN_CSV)
train_full["image"] = train_full.image.map(image_key)
if not train_full.image.is_unique:
    raise ValueError("Duplicate training images.")
train_pairs = get_pairs()
if train_pairs.shape != (team.NUM_SEEN_PAIRS, 2):
    raise ValueError("Unexpected training pair table.")
_, image_transform = get_transforms("imagenet")


def calibration_split(fit_frame, val_frame):
    """Hold out complete training pairs from the development training."""
    grouped = fit_frame.groupby("pair_row", sort=True)
    rows = grouped.agg(n=("image", "size"), crop=("crop", "first"), disease=("disease", "first"))
    crops = rows.crop.value_counts().to_dict()
    diseases = rows.disease.value_counts().to_dict()
    eligible = rows[rows.n.ge(5)].sample(frac=1, random_state=team.SEED + 1)
    held = []
    for pair_row, row in eligible.iterrows():
        if crops[row.crop] > 1 and diseases[row.disease] > 1:
            held.append(int(pair_row))
            crops[row.crop] -= 1
            diseases[row.disease] -= 1
        if len(held) == 10:
            break
    if len(held) < 10:
        raise ValueError("Ten train-derived holdouts with fit support were not available.")
    seen = val_frame.loc[~val_frame.pair_row.isin(held)].sample(frac=1, random_state=team.SEED + 2)
    cut = len(seen) // 2
    frames = {
        "tune_seen": seen.iloc[:cut],
        "tune_proxy": train_full.loc[train_full.pair_row.isin(held[:5])],
        "check_seen": seen.iloc[cut:],
        "check_proxy": train_full.loc[train_full.pair_row.isin(held[5:])],
    }
    if any(group.empty for group in frames.values()):
        raise ValueError("Calibration needs both seen and proxy images in each group.")
    fit_mask = torch.ones(len(train_pairs), dtype=torch.bool)
    fit_mask[held] = False
    return fit_mask, frames


settings = {
    'reference_commit': REFERENCE_COMMIT, 'backbone': BACKBONE, 'seed': team.SEED,
    'image_size': team.IMAGE_SIZE, 'resize_size': team.RESIZE_SIZE, 'val_fraction': team.VAL_FRACTION,
    'batch_size': BATCH_SIZE, 'epochs': EPOCHS, 'head_lr': HEAD_LR, 'backbone_lr': BACKBONE_LR,
    'weight_decay': WEIGHT_DECAY, 'optimizer': 'AdamW', 'schedule': 'CosineAnnealingLR',
    'label_smoothing': SMOOTHING, 'use_amp_cuda': AMP, 'decorrelation_weight': DECOR_WEIGHT,
    'supcon_weight': SUPCON_WEIGHT, 'temperature': TEMPERATURE,
    'blur_probability': BLUR_PROB, 'blur_sigma': BLUR_SIGMA,
    'recipes': RECIPES, 'calibration_grid': GAMMAS,
    'selection_rules': {'B0': 'seen validation Top-1', 'A1-A6': 'raw proxy harmonic mean'},
    'train_csv_sha256': sha256_file(team.TRAIN_CSV),
    'torch': str(torch.__version__), 'timm': timm.__version__, 'debug': DEBUG,
}


## 3. Model: same DINOv2 and classification heads, with controlled training losses

Each recipe starts from the **same pretrained DINOv2 ViT-B/14** and freshly seeded crop/disease linear heads. `depth=0` freezes the entire backbone; A4/A5/A6 only unfreeze the last 2/4/6 DINOv2 Transformer Blocks (the final norm remains frozen). All recipes use the same 323-candidate score: `log_softmax(crop) + log_softmax(disease)`. This is **not** the learned pairwise fusion scorer used by Approach 3C.

A1 adds centered-logit decorrelation; A2 adds cross-crop supervised contrastive learning on centered disease logits; A3 adds mild Gaussian blur. A4–A6 retain **all** A3 components unchanged, altering only the unfrozen depth as an experimental treatment (each recipe may select a different epoch under the same rule). Use separate raw and calibrated results to distinguish representation learning from score correction.


In [ ]:
class MultiTaskNet(nn.Module):
    """Latest Baseline 2 architecture."""
    def __init__(self, pretrained):
        super().__init__()
        self.backbone = timm.create_model(BACKBONE, pretrained=pretrained, num_classes=0,
                                          img_size=team.IMAGE_SIZE)
        self.backbone.requires_grad_(False)
        self.crop_head = nn.Linear(self.backbone.num_features, NUM_CROPS)
        self.disease_head = nn.Linear(self.backbone.num_features, NUM_DISEASES)

    def forward(self, images):
        features = self.backbone(images)
        return self.crop_head(features), self.disease_head(features)


def pair_scores(crop_logits, disease_logits, pairs):
    pairs = pairs.to(crop_logits.device)
    crop_log = F.log_softmax(crop_logits.float(), dim=1)
    disease_log = F.log_softmax(disease_logits.float(), dim=1)
    return crop_log[:, pairs[:, 0]] + disease_log[:, pairs[:, 1]]


def center_logits(logits):
    # Remove the per-image offset to which categorical predictions are invariant.
    return logits - logits.mean(dim=1, keepdim=True)


def decorrelation(crop, disease):
    def standardize(x):
        x = x - x.mean(0)
        return x / (x.square().mean(0) + 1e-4).sqrt()

    cross_correlation = standardize(crop).T @ standardize(disease) / len(crop)
    return cross_correlation.square().mean()


def cross_crop_supcon(embedding, crop, disease):
    z = F.normalize(embedding, dim=1)
    same_disease = disease[:, None].eq(disease[None, :])
    positives = same_disease & crop[:, None].ne(crop[None, :])
    negatives = ~same_disease
    similarity = z @ z.T / TEMPERATURE
    normalizer = similarity.masked_fill(~(positives | negatives), -1e9).logsumexp(dim=1)
    counts = positives.sum(dim=1)
    positive_mean = similarity.masked_fill(~positives, 0.0).sum(dim=1) / counts.clamp_min(1)
    usable = (counts > 0) & negatives.any(dim=1)
    per_anchor = (normalizer - positive_mean).masked_fill(~usable, 0.0)
    return per_anchor.sum() / usable.sum().clamp_min(1), usable.sum()


## 4. Development: B0→A3 add-ons, then A3→A6 fine-tuning depth

`DEBUG=True` runs every recipe for one epoch on debug data, but does **not** save official predictions or final results. With `DEBUG=False`, each run trains and selects seven independent recipes. B0's best epoch uses **common seen-validation Top-1**. A1–A6 select using raw harmonic mean on train-derived proxy-unseen pairs; γ is then selected for each proposed recipe from those same **tuning** groups. The independent check is descriptive only. B0 raw is the primary baseline reference.

**GPU note:** FT6 can exceed a 6GB laptop GPU. Uniform CUDA AMP and gradient checkpointing are enabled when possible; if an OOM persists, change **BATCH_SIZE for all seven recipes** and restart the comparison rather than silently reducing only FT6's batch.


In [ ]:
def make_model(depth, *, pretrained=True):
    model = MultiTaskNet(pretrained=pretrained)
    blocks = model.backbone.blocks
    if len(blocks) != 12 or depth not in (0, 2, 4, 6):
        raise ValueError('Expected 12-block DINOv2 ViT-B/14, with depth in {0,2,4,6}.')
    if depth:
        blocks[-depth:].requires_grad_(True)
        # Nonreentrant checkpointing is compatible with a frozen transformer prefix.
        try:
            from timm.layers import set_reentrant_ckpt
            set_reentrant_ckpt(False)
            model.backbone.set_grad_checkpointing(True)
        except (ImportError, AttributeError):
            print('Warning: timm gradient checkpointing not available; fine-tuning may need more VRAM.')
    return model.to(DEVICE).eval()


def make_optimizer(model):
    groups = [{'params': list(model.crop_head.parameters()) + list(model.disease_head.parameters()), 'lr': HEAD_LR}]
    tail = [p for p in model.backbone.parameters() if p.requires_grad]
    if tail:
        groups.append({'params': tail, 'lr': BACKBONE_LR})
    return torch.optim.AdamW(groups, weight_decay=WEIGHT_DECAY)


def make_loader(frame, transform, shuffle):
    return DataLoader(PlantDataset(frame, transform, 'imagenet'), batch_size=BATCH_SIZE,
                      shuffle=shuffle, num_workers=team.NUM_WORKERS, drop_last=False,
                      generator=torch.Generator().manual_seed(team.SEED))


def add_blur(images, generator, enabled):
    if not enabled:
        return images
    selected = torch.rand(len(images), generator=generator) < BLUR_PROB
    if not selected.any():
        return images
    selected = selected.to(images.device)
    result = images.clone()
    result[selected] = TF.gaussian_blur(images[selected], kernel_size=[5, 5],
                                        sigma=[BLUR_SIGMA, BLUR_SIGMA])
    return result


def train_epoch(model, optimizer, criterion, loader, recipe, stage, epoch, epochs, scaler):
    cfg = RECIPES[recipe]
    model.train()
    model.backbone.eval()  # Keep frozen prefix deterministic, while allowing gradient through trainable Blocks.
    if cfg['depth']:
        model.backbone.blocks[-cfg['depth']:].train()
    loss_total, samples, valid_anchors = 0., 0, 0
    blur_rng = torch.Generator().manual_seed(team.SEED + 172 + epoch)
    for step, (images, crop, disease, _) in enumerate(loader, 1):
        images, crop, disease = (batch.to(DEVICE) for batch in (images, crop, disease))
        images = add_blur(images, blur_rng, cfg['blur'])
        with torch.autocast(device_type='cuda' if AMP else 'cpu', dtype=torch.float16 if AMP else torch.bfloat16, enabled=AMP):
            crop_logits, disease_logits = model(images)
            crop_repr, disease_repr = center_logits(crop_logits.float()), center_logits(disease_logits.float())
            loss = criterion(crop_logits, crop) + criterion(disease_logits, disease)
            anchors = torch.tensor(0, device=DEVICE)
            if cfg['decor']:
                loss = loss + cfg['decor'] * decorrelation(crop_repr, disease_repr)
            if cfg['supcon']:
                contrast, anchors = cross_crop_supcon(disease_repr, crop, disease)
                loss = loss + cfg['supcon'] * contrast
        if not torch.isfinite(loss):
            raise FloatingPointError(f'Nonfinite loss in {recipe} ({stage})')
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        loss_total += float(loss.detach()) * len(images)
        samples += len(images)
        valid_anchors += int(anchors.detach())
        if step == 1 or step % 50 == 0 or step == len(loader):
            print(f'{recipe} | {stage} | epoch {epoch}/{epochs} | batch {step}/{len(loader)}', flush=True)
    if samples != len(loader.dataset):
        raise RuntimeError(f'{recipe}: fit loader skipped training images.')
    return {'epoch': epoch, 'images_trained': samples,
            'train_loss': loss_total / samples, 'supcon_anchor_fraction': valid_anchors / samples}


@torch.no_grad()
def development_scores(model, loader):
    model.eval()
    scores, labels = [], []
    for images, _, _, target in loader:
        c, d = model(images.to(DEVICE))
        scores.append(pair_scores(c, d, train_pairs).cpu())
        labels.append(target.cpu())
    return torch.cat(scores), torch.cat(labels)


def calibration_metrics(seen, proxy, fit_mask, gamma):
    seen_scores, y_seen = seen
    proxy_scores, y_proxy = proxy
    penalty = gamma * fit_mask.to(seen_scores.dtype)
    seen_scores = seen_scores - penalty
    proxy_scores = proxy_scores - penalty
    seen_top1 = top_k_accuracy(seen_scores, y_seen, 1)
    proxy_top1 = top_k_accuracy(proxy_scores, y_proxy, 1)
    return {'gamma': gamma, 'seen_top1': seen_top1, 'proxy_top1': proxy_top1,
            'seen_top5': top_k_accuracy(seen_scores, y_seen, 5),
            'proxy_top5': top_k_accuracy(proxy_scores, y_proxy, 5),
            'harmonic_top1': harmonic_mean(seen_top1, proxy_top1)}


def selection_key(row):
    return row['harmonic_top1'], row['proxy_top1'], row['seen_top1'], -row['epoch']


def snapshot_trainable(model):
    return {name: param.detach().cpu().clone()
            for name, param in model.named_parameters() if param.requires_grad}


@torch.no_grad()
def restore_trainable(model, state):
    for name, param in model.named_parameters():
        if param.requires_grad:
            param.copy_(state[name])


def develop_recipe(recipe, fit_frame, original_val_loader, groups, fit_mask):
    team.set_seed()
    cfg = RECIPES[recipe]
    depth = cfg['depth']
    model = make_model(depth)
    train_transform, _ = get_transforms('imagenet')
    fit_loader = make_loader(fit_frame, train_transform, shuffle=True)
    optimizer = make_optimizer(model)
    scaler = torch.amp.GradScaler('cuda', enabled=AMP)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    criterion = nn.CrossEntropyLoss(label_smoothing=SMOOTHING)
    history, best, best_weights, best_scores = [], None, None, None
    for epoch in range(1, EPOCHS + 1):
        row = {'recipe': recipe, 'unfrozen_blocks': depth, **cfg,
               **train_epoch(model, optimizer, criterion, fit_loader, recipe, 'development', epoch, EPOCHS, scaler)}
        scheduler.step()
        if not DEBUG:
            if recipe == BASELINE:
                s, sy = development_scores(model, original_val_loader)
                row['baseline_seen_val_top1'] = top_k_accuracy(s, sy, 1)
                key = (row['baseline_seen_val_top1'], -epoch)
            else:
                tune = [development_scores(model, groups[k]) for k in ('tune_seen','tune_proxy')]
                row.update(calibration_metrics(*tune, fit_mask, gamma=0.0))
                key = selection_key(row)
        else:
            key = (0.0, -epoch)
        if best is None or key > best['key']:
            best = {'row': row.copy(), 'key': key}
            best_weights = snapshot_trainable(model)
            if not DEBUG and recipe != BASELINE:
                best_scores = tune
        history.append(row)
        print({k: v for k, v in row.items() if k not in ('decor','supcon','blur')}, flush=True)
    output = DEVELOPMENT_DIR / recipe
    output.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(history).to_csv(output / 'training_history.csv', index=False)
    if DEBUG:
        del model
        return {'selection': best['row'], 'gamma': 0.0, 'independent_check': []}
    if recipe == BASELINE:
        gamma = 0.0
        check_rows = []
    else:
        grid = [calibration_metrics(*best_scores, fit_mask, gamma) for gamma in GAMMAS]
        gamma = max(grid, key=lambda row: (row['harmonic_top1'],row['proxy_top1'],
                                           row['seen_top1'],-row['gamma']))['gamma']
        restore_trainable(model, best_weights)
        check = [development_scores(model, groups[k]) for k in ('check_seen','check_proxy')]
        check_rows = [{'recipe': recipe,'scoring':rule,
                       **calibration_metrics(*check,fit_mask,penalty)}
                      for rule,penalty in (('raw',0.0),('calibrated',gamma))]
        pd.DataFrame(grid).to_csv(output / 'calibration_tune_grid.csv', index=False)
        pd.DataFrame(check_rows).to_csv(output / 'independent_check.csv', index=False)
    del model
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return {'selection': best['row'], 'gamma': gamma, 'independent_check': check_rows,
            'trainable_parameter_count': sum(v.numel() for v in best_weights.values())}


def run_development():
    team.set_seed()
    train_loader, val_loader, seen_loader, unseen_loader = get_loaders(BATCH_SIZE, norm='imagenet', debug=DEBUG)
    if seen_loader is not None or unseen_loader is not None:
        raise RuntimeError('Development loaders must not access official test data.')
    fit_full = train_loader.dataset.frame
    fit_mask, frames, groups = None, {}, {}
    fit_proposed = fit_full
    if not DEBUG:
        fit_mask, frames = calibration_split(fit_full, val_loader.dataset.frame)
        fit_allowed = fit_mask.nonzero(as_tuple=True)[0].tolist()
        fit_proposed = fit_full.loc[fit_full.pair_row.isin(fit_allowed)]
        groups = {key: make_loader(frame, image_transform, shuffle=False) for key,frame in frames.items()}
        forbidden = set(frames['tune_proxy'].image) | set(frames['check_proxy'].image)
        if forbidden & set(fit_proposed.image):
            raise RuntimeError('Proxy-unseen training leakage.')
    DEVELOPMENT_DIR.mkdir(parents=True, exist_ok=True)
    selections = {}
    for recipe in RECIPES:
        fit = fit_full if recipe == BASELINE else fit_proposed
        print(f'\n=== {recipe} | depth={RECIPES[recipe]["depth"]} | development images={len(fit)} ===',flush=True)
        selections[recipe] = develop_recipe(recipe, fit, val_loader, groups, fit_mask)
    if DEBUG:
        print('DEBUG complete: no full refit, official pairs or official labels have been accessed.')
        return {'recipes': selections}
    chosen = max(PROPOSALS, key=lambda recipe:(*selection_key(selections[recipe]['selection']),
                                                -RECIPES[recipe]['depth']))
    plan = {'primary_recipe':chosen, 'recipes':selections,
            'tune_pair_rows':sorted(frames['tune_proxy'].pair_row.unique().tolist()),
            'check_pair_rows':sorted(frames['check_proxy'].pair_row.unique().tolist()),
            'settings':settings, 'seen_pairs':train_pairs.tolist()}
    atomic_json(DEVELOPMENT_DIR / 'selection_plan.json',plan)
    table = pd.DataFrame([{'recipe':recipe,**value['selection'],'locked_gamma':value['gamma'],
                           'selected_proposal':recipe == chosen}
                          for recipe,value in selections.items()])
    table.to_csv(DEVELOPMENT_DIR / 'selected_models.csv',index=False)
    columns = ['recipe','unfrozen_blocks','epoch','baseline_seen_val_top1',
               'seen_top1','proxy_top1','harmonic_top1','supcon_anchor_fraction',
               'locked_gamma','selected_proposal']
    display(table.reindex(columns=columns))
    # This table (train-only) is the right place to inspect candidate improvements.
    # B0 has no proxy score because its common-split training sees the held-out training pairs.
    table.reindex(columns=columns).to_csv(DEVELOPMENT_DIR/'development_comparison.csv',index=False)
    print('Selected on raw proxy harmonic mean:',chosen)
    return plan


plan = run_development()


## 5. Fresh full-training refit (all seven recipes, each at its locked epoch)

Run this stage only after development. Every recipe starts from original pretrained DINOv2, freshly seeded classifier heads, and uses all 8,000 training CSV images. This is **not** continuing from the previously trained development checkpoint. Each final model uses the same selected recipe, selected epoch, and frozen hyperparameters; no official test information is accessed.

If `RUN_FULL_REFIT=True`, this runs automatically after development and saves separate checkpoints and SHA256 locks for B0–A6. The process is computationally expensive: you are training seven models for development **and** seven models for final refit. For report-ready comparisons, all seven official results should come from these comparable full-train refits.


In [ ]:
def read_final_lock(recipe):
    path = FINAL_DIR / recipe / 'selection_lock.json'
    checkpoint = CHECKPOINT_DIR / recipe / 'model.pt'
    if not path.is_file() or not checkpoint.is_file():
        raise FileNotFoundError(f'Missing final refit for {recipe}.')
    lock = json.loads(path.read_text(encoding='utf-8'))
    if (lock['settings'] != settings or lock['recipe'] != recipe
            or lock['selection_plan_sha256'] != sha256_file(DEVELOPMENT_DIR/'selection_plan.json')
            or lock['seen_pairs'] != train_pairs.tolist()
            or lock['checkpoint_sha256'] != sha256_file(checkpoint)):
        raise RuntimeError(f'Final checkpoint or selection plan changed for {recipe}.')
    return lock


def refit_recipe(recipe, selection):
    team.set_seed()
    cfg = RECIPES[recipe]
    model = make_model(cfg['depth'])
    transform, _ = get_transforms('imagenet')
    loader = make_loader(train_full, transform, shuffle=True)
    optimizer = make_optimizer(model)
    scaler = torch.amp.GradScaler('cuda', enabled=AMP)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    criterion = nn.CrossEntropyLoss(label_smoothing=SMOOTHING)
    epochs = int(selection['selection']['epoch'])
    if not 1 <= epochs <= EPOCHS:
        raise ValueError(f'Invalid locked refit epoch for {recipe}: {epochs}')
    history = []
    for epoch in range(1, epochs+1):
        row = train_epoch(model,optimizer,criterion,loader,recipe,'full_refit',epoch,epochs,scaler)
        scheduler.step()
        history.append(row)
    folder = FINAL_DIR / recipe
    checkpoint = CHECKPOINT_DIR / recipe / 'model.pt'
    folder.mkdir(parents=True,exist_ok=True)
    checkpoint.parent.mkdir(parents=True,exist_ok=True)
    pd.DataFrame(history).to_csv(folder/'refit_history.csv',index=False)
    refit = {'initialization':'original_pretrained_dinov2_fresh_heads',
             'data':'all_training_csv_rows','images_per_epoch':len(train_full),
             'epochs':epochs,'cosine_t_max':EPOCHS}
    metadata = {'settings':settings,'recipe':recipe,'config':cfg,'unfrozen_blocks':cfg['depth'],
                'seen_pairs':train_pairs.tolist(),'selection':selection['selection'],
                'gamma':float(selection['gamma']),'refit':refit}
    if hasattr(model.backbone,'set_grad_checkpointing'):
        model.backbone.set_grad_checkpointing(False)
    atomic_save(checkpoint,{'state_dict':model.cpu().state_dict(),**metadata})
    lock = {**metadata,'selection_plan_sha256':sha256_file(DEVELOPMENT_DIR/'selection_plan.json'),
            'checkpoint_sha256':sha256_file(checkpoint),'checkpoint_bytes':checkpoint.stat().st_size}
    atomic_json(folder/'selection_lock.json',lock)
    del model
    gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()
    return lock


def refit_full_train():
    if DEBUG:
        raise RuntimeError('Do not full-refit from debug mode.')
    if sha256_file(team.TRAIN_CSV)!=settings['train_csv_sha256']:
        raise RuntimeError('Training CSV changed after selection.')
    if json.loads((DEVELOPMENT_DIR/'selection_plan.json').read_text(encoding='utf-8'))!=plan:
        raise RuntimeError('Saved development plan differs from current selection.')
    finished={}
    for recipe in RECIPES:
        if (FINAL_DIR/recipe/'selection_lock.json').is_file():
            finished[recipe]=read_final_lock(recipe)
        else:
            finished[recipe]=refit_recipe(recipe,plan['recipes'][recipe])
        print('Locked final refit:',recipe,flush=True)
    return finished


runs = refit_full_train() if (not DEBUG and RUN_FULL_REFIT) else {}
if DEBUG:
    print('DEBUG finished. Set DEBUG=False for full development and optional final refit.')
elif not RUN_FULL_REFIT:
    print('Development finished; full refit skipped. To complete it, set RUN_FULL_REFIT=True and execute this cell.')

## 6. Official evaluation — separate manual opt-in, after all model locks

**Default: OFF.** This section is the only section that may access official `test_seen.csv` and `test_unseen.csv`. It first reads **filenames** and **10 unseen `(crop,disease)` identities**, predicts all 333 candidate scores and saves raw/calibrated predictions, and only afterward reads per-image `(crop,disease)` ground truth to calculate Top-1, Top-5, Overall, Harmonic Mean, and diagnostic metrics.

To run official evaluation: after `DEBUG=False` and all seven refit locks exist, change the `RUN_FINAL_EVALUATION=False` line **in the evaluation cell** to `True`, and run **only this evaluation cell and the following Save cell**. Do not rerun Setup/Development just to evaluate, since that starts a new timestamped run. The test split has already been reviewed elsewhere in this project; do not use these results to select or revise the method.


In [ ]:
def official_vocabulary():
    # The final shortlist uses only concept identities and discards per-image row order.
    vocabulary = pd.read_csv(team.TEST_UNSEEN_CSV, header=None, usecols=[1, 2],
                             dtype=str, keep_default_na=False)
    vocabulary.columns = ["crop", "disease"]
    if not vocabulary.empty and not vocabulary.iloc[0]["crop"].strip().isdigit():
        vocabulary = vocabulary.iloc[1:]
    vocabulary = (vocabulary.astype("int64").drop_duplicates()
                  .sort_values(["crop", "disease"]))
    extra_pairs = torch.as_tensor(vocabulary.to_numpy(copy=True), dtype=torch.long)
    pairs = torch.cat((train_pairs, extra_pairs))
    if (len(extra_pairs) != team.NUM_UNSEEN_PAIRS or len(pairs.unique(dim=0)) != team.NUM_PAIRS
            or not set(extra_pairs[:, 0].tolist()) <= set(train_pairs[:, 0].tolist())
            or not set(extra_pairs[:, 1].tolist()) <= set(train_pairs[:, 1].tolist())):
        raise ValueError("Expected distinct official unseen pairs composed of training-derived concepts.")
    return pairs


def official_filenames(path):
    values = pd.read_csv(path, header=None, usecols=[0], dtype=str, keep_default_na=False).iloc[:, 0].tolist()
    header = {"image", "image_path", "image_name", "filename", "file_name", "file_path", "path"}
    if values and (values[0].strip().lower().replace(" ", "_") in header or
                   (len(values) > 1 and not Path(values[0]).suffix and Path(values[1]).suffix)):
        values = values[1:]
    names = [image_key(name) for name in values]
    if not names or len(set(names)) != len(names):
        raise ValueError(f"Expected unique image names in {path.name}.")
    return names


class OfficialImages(Dataset):
    def __init__(self, names):
        self.names = names

    def __len__(self):
        return len(self.names)

    def __getitem__(self, index):
        with Image.open(team.IMAGE_DIR / self.names[index]) as image:
            return image_transform(image.convert("RGB"))


def official_targets(csv_path, filenames, row_by_pair, *, seen):
    # This per-image projection is used after predictions. Ignore the official pair_label column.
    frame = pd.read_csv(csv_path, header=None, usecols=[0, 1, 2], dtype=str, keep_default_na=False)
    frame.columns = ["image", "crop", "disease"]
    if not frame.empty and not frame.iloc[0]["crop"].strip().isdigit():
        frame = frame.iloc[1:]
    if frame.image.map(image_key).tolist() != filenames:
        raise ValueError("Official targets and predictions have different image orders.")
    keys = frame[["crop", "disease"]].astype("int64").itertuples(index=False, name=None)
    try:
        labels = torch.tensor([row_by_pair[key] for key in keys], dtype=torch.long)
    except KeyError as error:
        raise ValueError(f"Official ground-truth pair absent from the frozen candidate table: {error}") from error
    if not ((labels < team.NUM_SEEN_PAIRS) == seen).all():
        raise ValueError("Official test split does not match seen membership.")
    return labels


def top_predictions(scores, offset=0):
    return {k: scores.topk(k, dim=1).indices + offset for k in (1, 5)}


def prediction_accuracy(predictions, labels):
    return round(100 * predictions.eq(labels[:, None]).any(1).float().mean().item(), 2)


@torch.inference_mode()
def predict_official(model, pairs, filenames, split, perturb=None):
    loader = DataLoader(OfficialImages(filenames), batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=0, generator=torch.Generator().manual_seed(team.SEED))
    mean, std = (torch.tensor(v).view(1, 3, 1, 1) for v in team.NORM_STATS["imagenet"])
    parts, concepts = [], []
    for step, images in enumerate(loader, 1):
        if perturb is not None:
            images = (perturb(images * std + mean) - mean) / std
        crop_logits, disease_logits = model(images.to(DEVICE))
        scores = pair_scores(crop_logits, disease_logits, pairs).cpu()
        if perturb is None:
            concepts.append(torch.stack((crop_logits.argmax(1), disease_logits.argmax(1)), dim=1).cpu())
        if scores.shape != (len(images), len(pairs)) or not torch.isfinite(scores).all():
            raise ValueError("Unexpected official candidate predictions.")
        parts.append(scores)
        if step == 1 or step % 50 == 0 or step == len(loader):
            print(f"{split}: batch {step}/{len(loader)}", flush=True)
    return torch.cat(parts), torch.cat(concepts) if concepts else None



def predict_recipe(recipe, run, pairs, filenames):
    saved = torch.load(CHECKPOINT_DIR / recipe / "model.pt", map_location="cpu", weights_only=True)
    for key in ("settings", "recipe", "unfrozen_blocks", "selection", "gamma", "seen_pairs", "refit"):
        if saved[key] != run[key]:
            raise RuntimeError(f"Checkpoint and selection lock differ for {recipe}: {key}.")
    model = MultiTaskNet(pretrained=False)
    model.load_state_dict(saved["state_dict"], strict=True)
    model = model.to(DEVICE).eval()
    boundary = len(train_pairs)
    membership = torch.arange(len(pairs)) < boundary
    clean, heads = zip(*(predict_official(model, pairs, names, split)
                         for names, split in zip(filenames, ("seen", "unseen"))))
    perturbed = {}
    if RUN_ROBUSTNESS:
        with torch.random.fork_rng(devices=[]):
            torch.random.default_generator.manual_seed(team.SEED)
            for condition, transform in PERTURBATIONS.items():
                perturbed[condition], _ = predict_official(model, pairs, filenames[0], condition, transform)

    predictions = {}
    rules = [("raw", 0.0)] if recipe == BASELINE else [("raw", 0.0), ("calibrated", run["gamma"])]
    for rule, gamma in rules:
        seen_scores, unseen_scores = (scores - gamma * membership for scores in clean)
        predictions[rule] = {
            "gamma": gamma,
            "seen": top_predictions(seen_scores),
            "unseen": top_predictions(unseen_scores),
            "seen_only": top_predictions(seen_scores[:, :boundary]),
            "unseen_only": top_predictions(unseen_scores[:, boundary:], offset=boundary),
            "robust": {name: (scores - gamma * membership).topk(1, dim=1).indices
                       for name, scores in perturbed.items()},
        }

    return predictions, clean, heads


def score_trial(run, pairs, forward_pass, y_seen, y_unseen):
    predictions, clean, (seen_heads, unseen_heads) = forward_pass
    boundary = len(train_pairs)
    y_all = torch.cat((y_seen, y_unseen))

    reports, ranking, per_pair, robustness, restricted = [], [], [], [], []
    same_crop_candidates = pairs[:, 0].eq(pairs[y_unseen, :1])
    seen_head_hits = seen_heads.eq(pairs[y_seen])
    unseen_head_hits = unseen_heads.eq(pairs[y_unseen])
    head_top1 = {
        f"{split}_{head}_evidence_top1": round(100 * matches[:, col].float().mean().item(), 2)
        for split, matches in (("seen", seen_head_hits), ("unseen", unseen_head_hits))
        for col, head in enumerate(("crop", "disease"))
    }
    for name, prediction in predictions.items():
        gamma = prediction["gamma"]
        raw_scores = clean[1]
        crop_oracle = raw_scores.masked_fill(~same_crop_candidates, -torch.inf).topk(1, dim=1).indices
        raw_margins = raw_scores.gather(1, y_unseen[:, None]).squeeze(1) - raw_scores[:, :boundary].amax(dim=1)
        final_margins = raw_margins + gamma
        only_correct = prediction["unseen_only"][1].squeeze(1).eq(y_unseen)
        metrics = {"scoring": name, "gamma": gamma, "unfrozen_blocks": run["unfrozen_blocks"],
                   "selected_on_tune": run["recipe"] == plan["primary_recipe"],
                   "epoch": run["selection"]["epoch"], "candidates": len(pairs), **head_top1,
                   "raw_true_crop_candidate_top1": prediction_accuracy(crop_oracle, y_unseen)}
        for k in (1, 5):
            metrics[f"seen_top{k}"] = prediction_accuracy(prediction["seen"][k], y_seen)
            metrics[f"unseen_top{k}"] = prediction_accuracy(prediction["unseen"][k], y_unseen)
            joined = torch.cat((prediction["seen"][k], prediction["unseen"][k]))
            metrics[f"overall_top{k}"] = prediction_accuracy(joined, y_all)
        metrics["harmonic_mean"] = harmonic_mean(metrics["seen_top1"], metrics["unseen_top1"])
        metrics["abs_seen_unseen_gap_pp"] = round(abs(metrics["seen_top1"] - metrics["unseen_top1"]), 2)
        seen_winner, unseen_winner = (prediction[split][1].squeeze(1) for split in ("seen", "unseen"))
        metrics["seen_predicted_unseen"] = int((seen_winner >= boundary).sum())
        for split, winner, target in (("seen", seen_winner, y_seen), ("unseen", unseen_winner, y_unseen)):
            concept_hits = pairs[winner].eq(pairs[target]).float().mean(dim=0) * 100
            metrics[f"{split}_test_crop_acc"], metrics[f"{split}_test_disease_acc"] = [
                round(float(value), 2) for value in concept_hits]
        reports.append(metrics)

        correct = unseen_winner.eq(y_unseen)
        predicts_seen = unseen_winner < boundary
        blocked = only_correct & predicts_seen
        same_crop = pairs[unseen_winner, 0].eq(pairs[y_unseen, 0])
        ranking.append({"scoring": name, "unseen_images": len(y_unseen), "joint_correct": int(correct.sum()),
                        "unseen_only_correct": int(only_correct.sum()), "unseen_predicted_seen": int(predicts_seen.sum()),
                        "blocked_by_seen": int(blocked.sum()), "blocked_same_crop": int((blocked & same_crop).sum()),
                        "blocked_other_crop": int((blocked & ~same_crop).sum()),
                        "median_true_pair_minus_top_seen_score": float(final_margins.median())})
        for index in range(boundary, len(pairs)):
            mask = y_unseen.eq(index)
            crop, disease = pairs[index].tolist()
            per_pair.append({"scoring": name, "crop": crop, "disease": disease, "support": int(mask.sum()),
                             "joint_correct": int((mask & correct).sum()),
                             "unseen_only_correct": int((mask & only_correct).sum()),
                             "blocked_by_seen": int((mask & blocked).sum())})
        for key, predictions_at_one in prediction["robust"].items():
            accuracy = prediction_accuracy(predictions_at_one, y_seen)
            robustness.append({"scoring": name, "condition": key, "seen_top1": accuracy,
                               "drop_pp": round(metrics["seen_top1"] - accuracy, 2)})
        for split, targets, count in (("seen_only", y_seen, boundary),
                ("unseen_only", y_unseen, len(pairs) - boundary)):
            restricted.append({"scoring": name, "split": split, "candidates": count,
                               **{f"top{k}": prediction_accuracy(prediction[split][k], targets) for k in (1, 5)}})
    return ({"results": reports, "restricted_candidates": restricted, "robustness": robustness,
             "unseen_ranking_diagnostics": ranking, "unseen_per_pair": per_pair},
            {name: pred["seen"][1].squeeze(1) for name, pred in predictions.items()})



def depth_comparisons(rows):
    # Fine-tuning depths 2,4,6 must be compared to A3 at the SAME scoring rule.
    reference = {row['scoring']:row for row in rows if row['recipe']=='A3_Add_Blur'}
    output=[]
    for row in rows:
        if row['recipe'] not in FINE_TUNES:
            continue
        r=reference[row['scoring']]
        output.append({'recipe':row['recipe'],'scoring':row['scoring'],
                       'reference':'A3_Add_Blur','unfrozen_blocks':row['unfrozen_blocks'],
                       **{f'{metric}_gain_pp':round(row[metric]-r[metric],2)
                          for metric in ('seen_top1','unseen_top1','harmonic_mean')},
                       'abs_seen_unseen_gap_pp':row['abs_seen_unseen_gap_pp']})
    return output


def additive_comparisons(rows):
    # Each adjacent difference changes one add-on and has frozen backbone (depth 0).
    chain=['B0_Baseline2','A1_Decorrelation','A2_Add_SupCon','A3_Add_Blur']
    lookup={(row['recipe'],row['scoring']):row for row in rows}
    diffs=[]
    for scoring in ('raw','calibrated'):
        for before,after in zip(chain,chain[1:]):
            if (before,scoring) not in lookup or (after,scoring) not in lookup: continue
            x,y=lookup[(before,scoring)],lookup[(after,scoring)]
            diffs.append({'reference':before,'recipe':after,'scoring':scoring,
                          **{f'{key}_gain_pp':round(y[key]-x[key],2)
                             for key in ('seen_top1','unseen_top1','harmonic_mean')}})
    return diffs

def official_evaluation():
    if DEBUG or not runs or any(read_final_lock(recipe) != runs[recipe] for recipe in RECIPES):
        raise RuntimeError("Complete development and refit for every depth before official evaluation.")
    pairs = official_vocabulary()
    paths = (team.TEST_SEEN_CSV, team.TEST_UNSEEN_CSV)
    filenames = tuple(official_filenames(path) for path in paths)
    if set(filenames[0]) & set(filenames[1]) or set(train_full.image) & (set(filenames[0]) | set(filenames[1])):
        raise ValueError("Training and official split filenames overlap.")
    predictions = {recipe: predict_recipe(recipe, runs[recipe], pairs, filenames) for recipe in RECIPES}
    # No per-image labels are retrieved until every locked rule has predicted.
    rows = {tuple(pair): i for i, pair in enumerate(pairs.tolist())}
    y_seen = official_targets(paths[0], filenames[0], rows, seen=True)
    y_unseen = official_targets(paths[1], filenames[1], rows, seen=False)
    summary = {"selection": runs, "recipe_selected_on_tune": plan["primary_recipe"],
               "samples": {"seen": len(y_seen), "unseen": len(y_unseen)},
               "pair_order": "train_then_concept_ids", "candidate_pairs": pairs.tolist()}
    winners = {}
    for recipe, prediction in predictions.items():
        report, seen_winners = score_trial(runs[recipe], pairs, prediction, y_seen, y_unseen)
        winners.update({f"{recipe}_{rule}": values for rule, values in seen_winners.items()})
        for key, values in report.items():
            for row in values:
                row["recipe"] = recipe
            summary.setdefault(key, []).extend(values)
    summary["depth_gains"] = depth_comparisons(summary["results"])
    summary["additive_gains"] = additive_comparisons(summary["results"])
    return summary, winners, y_seen


RUN_ROBUSTNESS = False        # Set True to run brightness/blur/noise (extra inference time)
RUN_FINAL_EVALUATION = False   # Switch True ONLY after all seven model locks exist.
evaluation = official_evaluation() if RUN_FINAL_EVALUATION else None
if not RUN_FINAL_EVALUATION:
    print('Official evaluation skipped. No official test metadata or labels were accessed.')


## 7. Export official comparison and interpret gains

After **manually** enabling Official Evaluation, this cell writes a single `official_comparison.csv` with B0 and A1–A6 (raw + calibrated for proposed recipes), a `depth_gains.csv` comparing A4/A5/A6 **against A3 with the same scoring rule**, and an `additive_gains.csv` for B0→A1→A2→A3. It also writes per-recipe diagnostics and an `official_report.json`.

**Ablation interpretation:** Only the raw rows speak to representation/ranking improvements without score calibration; show calibrated rows as a **separate scoring condition**. Repeated official tests must not be used for hyperparameter tuning or to decide which recipe to keep.


In [ ]:
def save_results(summary, seen_winners, seen_targets):
    OFFICIAL_DIR.mkdir(parents=True, exist_ok=True)
    # Align the team confusion plot order only after predictions and metrics are fixed.
    pairs = [tuple(pair) for pair in summary["candidate_pairs"]]
    lookup = {tuple(pair): row for row, pair in enumerate(get_pairs(final_evaluation=True).tolist())}
    if set(pairs) != lookup.keys():
        raise ValueError("Candidate identities differ from the common evaluator.")
    to_common = torch.tensor([lookup[pair] for pair in pairs])
    canonical_targets = to_common[seen_targets]
    if (canonical_targets >= len(train_pairs)).any():
        raise ValueError("The common confusion plot expects seen target rows.")
    for name, winners in seen_winners.items():
        votes = F.one_hot(to_common[winners], num_classes=len(pairs)).float()
        stem = (OFFICIAL_DIR / name).relative_to(team.RESULTS_DIR).as_posix()
        save_confusion(votes, canonical_targets, stem, final_evaluation=True)

    table = pd.DataFrame(summary["results"])
    table.to_csv(OFFICIAL_DIR / "official_comparison.csv", index=False)
    for name in ("depth_gains", "additive_gains", "restricted_candidates", "robustness", "unseen_ranking_diagnostics", "unseen_per_pair"):
        pd.DataFrame(summary[name]).to_csv(OFFICIAL_DIR / f"{name}.csv", index=False)
    for row in summary["results"]:
        atomic_json(OFFICIAL_DIR / f"{row['recipe']}_{row['scoring']}.json", row)
    atomic_json(OFFICIAL_DIR / "official_report.json", summary)
    return table


if evaluation is not None:
    table = save_results(*evaluation)
    display(table[["recipe", "unfrozen_blocks", "scoring", "gamma", "epoch", "selected_on_tune",
                   "seen_top1", "unseen_top1", "unseen_top5", "overall_top1", "harmonic_mean"]])
    display(pd.DataFrame(evaluation[0]["depth_gains"]))
    display(pd.DataFrame(evaluation[0]["additive_gains"]))
    for recipe in RECIPES:
        print(recipe, "checkpoint:", (CHECKPOINT_DIR / recipe / "model.pt").resolve())
    print("Official results:", OFFICIAL_DIR.resolve())
else:
    print('Development results:', DEVELOPMENT_DIR.resolve())
    if runs:
        print('Final refit locks:', FINAL_DIR.resolve())
    print('Official evaluation is OFF unless enabled in the evaluation cell.')
